# Housing Data Cleaning Pipeline
This notebook merges four regional housing datasets and applies a full data cleaning pipeline — handling missing values, dropping irrelevant columns, and computing derived fields,to produce a single analysis ready CSV.

## 1. Import Libraries

In [ ]:
import pandas as pd

## 2. Load Data
Four CSV files sourced from Redfin are loaded and previewed.

In [ ]:
df1 = pd.read_csv('/content/housingdata1.csv')
df2 = pd.read_csv('/content/housingdata2.csv')
df3 = pd.read_csv('/content/housingdata3.csv')
df4 = pd.read_csv('/content/housingdata4.csv')

df1.head()

### Inspect column names across all files

In [ ]:
for i, df in enumerate([df1, df2, df3, df4], start=1):
    print(f"df{i} columns:", df.columns.tolist())

## 3. Combine Datasets

In [ ]:
# Stack all four dataframes vertically (row-wise)
df_all = pd.concat([df1, df2, df3, df4], axis=0).reset_index(drop=True)

# Save raw combined data as a checkpoint
df_all.to_csv('HousingData_all.csv', index=False)

df_all.head()

## 4. Exploratory Overview

In [ ]:
# Data types and non-null counts
df_all.info()

In [ ]:
# Summary statistics for numeric columns
df_all.describe()

In [ ]:
# Count of unique cities in the dataset
print(f"Unique cities: {df_all['CITY'].nunique()}")
df_all['CITY'].unique()

## 5. Drop Irrelevant Columns
Columns that are not useful for analysis are removed.

In [ ]:
cols_to_drop = [
    'DAYS ON MARKET', 'STATUS',
    'NEXT OPEN HOUSE START TIME', 'NEXT OPEN HOUSE END TIME',
    'SALE TYPE',
    'URL (SEE https://www.redfin.com/buy-a-home/comparative-market-analysis FOR INFO ON PRICING)',
    'SOURCE', 'MLS#', 'LOCATION'
]

df_all = df_all.drop(columns=cols_to_drop)
df_all.head()

## 6. Handle Missing Values

### 6a. Assess missing data

In [ ]:
# Percentage of missing values per column, sorted descending
missing_pct = df_all.isnull().mean() * 100
missing_pct.sort_values(ascending=False)

### 6b. Fill HOA/MONTH — missing means no HOA fee

In [ ]:
df_all['HOA/MONTH'] = df_all['HOA/MONTH'].fillna(0)
df_all['HOA/MONTH'].isnull().sum()

### 6c. Fill SOLD DATE — mark unsold listings as 'Unknown'

In [ ]:
df_all['SOLD DATE'] = df_all['SOLD DATE'].fillna('Unknown')

# Preview rows where no sale date exists
df_all[df_all['SOLD DATE'] == 'Unknown'].head()

### 6d. Fill YEAR BUILT and LOT SIZE with mode (most common value)

In [ ]:
for col in ['YEAR BUILT', 'LOT SIZE']:
    df_all[col] = df_all[col].fillna(df_all[col].mode()[0])

### 6e. Fill BEDS and BATHS with median (robust to outliers)

In [ ]:
for col in ['BEDS', 'BATHS']:
    df_all[col] = df_all[col].fillna(df_all[col].median())

### 6f. Fill SQUARE FEET using group-based imputation by property type
Different property types have very different sizes, so we impute within each group.

In [ ]:
# Group mean imputation by PROPERTY TYPE
df_all['SQUARE FEET'] = df_all.groupby('PROPERTY TYPE')['SQUARE FEET'].transform(
    lambda x: x.fillna(x.mean())
)

# Fill any remaining nulls (e.g. Vacant Land with no sqft data) with 0
df_all['SQUARE FEET'] = df_all['SQUARE FEET'].fillna(0)

### 6g. Compute missing $/SQUARE FEET from PRICE and SQUARE FEET

In [ ]:
df_all['$/SQUARE FEET'] = df_all['$/SQUARE FEET'].fillna(
    df_all['PRICE'] / df_all['SQUARE FEET']
)

### Final missing value check

In [ ]:
df_all.isnull().sum()

## 7. Export Cleaned Dataset

In [ ]:
df_all.to_csv('HousingData_cleaned.csv', index=False)
print(f"Cleaned dataset saved: {df_all.shape[0]} rows, {df_all.shape[1]} columns")